# A2: GPU basics: bandwidth, matmul throughput, roofline

Three ideas, no deeper:
1. **Decode is limited by memory bandwidth.** Each generated token must read all ~2.2 GB of weights. We measure bandwidth once.
2. **Prefill is limited by compute.** Many tokens share one weight read, so the matmul units become the bottleneck. We measure matmul throughput once.
3. **Roofline.** A kernel does `FLOPs` of work and moves `bytes`. Its *arithmetic intensity* is `FLOPs / bytes`. Below the *ridge point* (peak FLOPs / peak bandwidth) it is memory-bound, above it compute-bound. For a decode matmul `(B, K) @ (K, N)` the weight dominates the bytes, so intensity is about `B`: **batching raises intensity**, and that is why it helps.

The T4 varies run to run (clocks, power, temperature), so we report **ranges** (min / median / max over repeated trials), not single numbers.

In [ ]:
# Setup: clone or pull the repo, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
import importlib, json
import torch
import tinyserve.roofline, tinyserve.gpu, tinyserve.plots
for module in (tinyserve.roofline, tinyserve.gpu, tinyserve.plots):
    importlib.reload(module)  # picks up repo edits after a git pull
from tinyserve.envinfo import collect_env

DEVICE = "cuda"
print(collect_env())
# Footnote: clocks and power explain run-to-run variation on the T4.
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu", "--format=csv"],
    capture_output=True, text=True).stdout)

## 1. Memory bandwidth
We time a device-to-device copy of `S` bytes. It reads `S` and writes `S`, so traffic is `2S`. Small buffers fit in the T4's ~4 MiB L2 cache and look unrealistically fast; large buffers measure the real GDDR6 bandwidth. Datasheet peak: **320 GB/s**.

In [ ]:
from tinyserve.gpu import measure_bandwidth_curve
from tinyserve.roofline import T4_DATASHEET

bandwidth_rows = measure_bandwidth_curve([1, 4, 16, 64, 256, 1024], DEVICE)
for row in bandwidth_rows:
    stats = row["gb_s"]
    print(f"{row['size_mib']:>5} MiB  {stats['min']:7.1f} / {stats['median']:7.1f} / {stats['max']:7.1f} GB/s  (min/median/max)")

# Large buffers only: these are not served by L2.
MEASURED_BANDWIDTH = max(r["gb_s"]["median"] for r in bandwidth_rows if r["size_mib"] >= 64) * 1e9
print(f"\nmeasured bandwidth: {MEASURED_BANDWIDTH / 1e9:.0f} GB/s  ({MEASURED_BANDWIDTH / T4_DATASHEET['bandwidth_bytes_per_s']:.0%} of the 320 GB/s datasheet)")

## 2. Matmul throughput
A big square matmul does `2*n^3` FLOPs on `3*n^2` elements, so its intensity grows with `n`: it is compute-bound. FP16 uses tensor cores (datasheet **65 TFLOPS**); FP32 does not (datasheet **8.1 TFLOPS**). Real T4s typically land well below the FP16 peak because of power and clock limits.

In [ ]:
from tinyserve.gpu import measure_matmul_curve

fp16_rows = measure_matmul_curve([512, 1024, 2048, 4096, 8192], DEVICE, torch.float16)
fp32_rows = measure_matmul_curve([1024, 2048, 4096], DEVICE, torch.float32)
for name, rows in (("fp16", fp16_rows), ("fp32", fp32_rows)):
    for row in rows:
        stats = row["tflops"]
        print(f"{name} n={row['size']:>5}  intensity {row['intensity']:7.1f}  {stats['min']:5.1f} / {stats['median']:5.1f} / {stats['max']:5.1f} TFLOPS")

MEASURED_FP16_FLOPS = max(r["tflops"]["median"] for r in fp16_rows) * 1e12
MEASURED_FP32_FLOPS = max(r["tflops"]["median"] for r in fp32_rows) * 1e12
print(f"\nmeasured peak: fp16 {MEASURED_FP16_FLOPS / 1e12:.1f} TFLOPS ({MEASURED_FP16_FLOPS / T4_DATASHEET['fp16_flops']:.0%} of datasheet), "
      f"fp32 {MEASURED_FP32_FLOPS / 1e12:.1f} TFLOPS ({MEASURED_FP32_FLOPS / T4_DATASHEET['fp32_flops']:.0%})")

## 3. The roofline and the decode ceiling
- **Ridge point** = peak FLOPs / peak bandwidth: the intensity needed to keep the matmul units busy.
- **Decode ceiling** at batch 1 = bandwidth / weight bytes. This ignores the KV cache and overheads, so real decode is slower. TinyLlama FP16 weights are 2,200,096,768 bytes (from the A1 results).

In [ ]:
from tinyserve.roofline import ridge_point, decode_tokens_per_s_ceiling

WEIGHT_BYTES = json.load(open("results/A1_shapes.json"))["data"]["weight_bytes"]
print("ridge, datasheet:", round(ridge_point(T4_DATASHEET["fp16_flops"], T4_DATASHEET["bandwidth_bytes_per_s"])), "FLOPs/byte")
print("ridge, measured :", round(ridge_point(MEASURED_FP16_FLOPS, MEASURED_BANDWIDTH)), "FLOPs/byte")
print()
for batch in (1, 8, 32, 64):
    ceiling = decode_tokens_per_s_ceiling(WEIGHT_BYTES, MEASURED_BANDWIDTH, batch)
    print(f"batch {batch:>3}: memory-bound decode ceiling {ceiling:7.0f} tokens/s total, {ceiling / batch:6.1f} tokens/s per sequence")

## 4. Why batching helps: one decode-sized matmul
TinyLlama's MLP up-projection is `(B, 2048) @ (2048, 5632)` in FP16. The weight is 23 MB, so it is read from memory (it does not fit in L2). Watch what happens as `B` grows: time stays almost flat while memory-bound, so rows per second scale nearly linearly with `B`. Once intensity passes the ridge point, time starts to grow.

In [ ]:
from tinyserve.gpu import measure_decode_curve

decode_rows = measure_decode_curve([1, 2, 4, 8, 16, 32, 64, 128, 256, 512], 2048, 5632, DEVICE, torch.float16)
print(f"{'B':>4} {'ms':>8} {'intensity':>10} {'GB/s':>8} {'TFLOPS':>8} {'rows/s':>10}")
for row in decode_rows:
    print(f"{row['batch']:>4} {row['median_ms']:8.4f} {row['intensity']:10.1f} {row['gb_s']:8.1f} {row['tflops']:8.2f} {row['rows_per_s']:10.0f}")

## 5. The roofline diagram (code, not a screenshot)
Black line: the roofs using our *measured* peaks. Points: big matmuls sit on the flat compute roof; decode-sized matmuls climb the slanted memory roof as `B` grows.

In [ ]:
from IPython.display import Image, display
from tinyserve.plots import plot_roofline

points = [(f"B={r['batch']}", r["intensity"], r["tflops"] * 1e12) for r in decode_rows]
points += [(f"n={r['size']}", r["intensity"], r["tflops"]["median"] * 1e12) for r in fp16_rows]
os.makedirs("results", exist_ok=True)
plot_path = plot_roofline(MEASURED_FP16_FLOPS, MEASURED_BANDWIDTH, points, "results/A2_roofline.png",
                          title="T4 FP16 roofline (measured peaks)")
display(Image(plot_path))

## 6. Save results and run the tests

In [ ]:
from tinyserve.envinfo import save_result

saved_path = save_result("A2_gpu", {
    "datasheet": T4_DATASHEET,
    "measured_bandwidth_gb_s": MEASURED_BANDWIDTH / 1e9,
    "measured_fp16_tflops": MEASURED_FP16_FLOPS / 1e12,
    "measured_fp32_tflops": MEASURED_FP32_FLOPS / 1e12,
    "bandwidth_curve": bandwidth_rows,
    "matmul_fp16": fp16_rows,
    "matmul_fp32": fp32_rows,
    "decode_matmul_2048x5632_fp16": decode_rows,
})
print(saved_path)
print(json.dumps(json.load(open(saved_path))["data"]["decode_matmul_2048x5632_fp16"][:2], indent=1))

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_a2_roofline.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3500:], result.stderr[-1000:])

In [ ]:
# Print the full results file so it can be copied into the repo from the saved notebook.
print(open(saved_path).read())